# Оптимизация PyTorch

In [1]:
from pathlib import Path
import os
import sys
import json
import zipfile

base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
root = base / 'trompt'
root.mkdir(exist_ok=True)
os.chdir(root)
Path('tests').mkdir(exist_ok=True)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [2]:
%%writefile train.py


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data

import os
import urllib.request
from tqdm import tqdm


class TromptCell(nn.Module):
    def __init__(self, n_columns, n_prompts, d_model):
        super().__init__()

        self.feature_emb_weight = nn.Parameter(torch.empty(n_columns, d_model))
        self.feature_emb_bias = nn.Parameter(torch.empty(n_columns, d_model))
        self.ln_emb = nn.LayerNorm(d_model)


        self.ln_col = nn.LayerNorm(d_model)
        self.ln_prompt = nn.LayerNorm(d_model)
        self.dense_imp = nn.Linear(2 * d_model, d_model)

        self.emb_column = nn.Parameter(torch.empty(n_columns, d_model))
        self.emb_prompt = nn.Parameter(torch.empty(n_prompts, d_model))


        self.dense_expand = nn.Linear(1, n_prompts)

        self.reset_parameters()

    def reset_parameters(self):
        d_rsqrt = self.feature_emb_weight.shape[1] ** -0.5
        nn.init.uniform_(self.feature_emb_weight, -d_rsqrt, d_rsqrt)
        nn.init.uniform_(self.feature_emb_bias, -d_rsqrt, d_rsqrt)
        nn.init.normal_(self.emb_column, std=0.01)
        nn.init.normal_(self.emb_prompt, std=0.01)

    def forward(self, x: torch.Tensor, prev_cell_out: torch.Tensor) -> torch.Tensor:
        x_emb = x.unsqueeze(-1) * self.feature_emb_weight + self.feature_emb_bias.unsqueeze(0)
        x_emb = F.relu(x_emb)
        x_emb = self.ln_emb(x_emb)

        x_prompt = self.emb_prompt.unsqueeze(0).repeat(x_emb.shape[0], 1, 1)
        x_prompt = self.dense_imp(torch.cat([self.ln_prompt(x_prompt), prev_cell_out], dim=-1)) + x_prompt
        x_column = self.ln_col(self.emb_column.unsqueeze(0).repeat(x_emb.shape[0], 1, 1))
        mask = torch.softmax(x_prompt @ x_column.transpose(1, 2), dim=-1)

        x_emb = x_emb.unsqueeze(1) + self.dense_expand(x_emb.unsqueeze(-1)).permute(0, 3, 1, 2)
        x_out = (mask.unsqueeze(-1) * x_emb).sum(dim=2)
        return x_out


class TromptDownstream(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.dense0 = nn.Linear(d_model, 1)
        self.dense1 = nn.Linear(d_model, d_model)
        self.ln = nn.LayerNorm(d_model)
        self.dense_out = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pw = torch.softmax(self.dense0(x).squeeze(-1), dim=-1)
        xnew = (pw.unsqueeze(-1) * x).sum(dim=-2)
        return self.dense_out(self.ln(F.relu(self.dense1(xnew))))


class Trompt(nn.Module):
    def __init__(self, n_columns, n_prompts, d_model, n_cycles):
        super().__init__()
        self.tcells = nn.ModuleList([TromptCell(n_columns, n_prompts, d_model) for _ in range(n_cycles)])
        self.tdown = TromptDownstream(d_model)
        self.prompt = nn.Parameter(torch.empty(n_prompts, d_model))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(self.prompt, std=0.01)

    def forward(self, x):
        x_prompt = self.prompt.unsqueeze(0).repeat(x.shape[0], 1, 1)
        outputs = []
        for cell in self.tcells:
            outputs.append(self.tdown(cell(x, x_prompt)))
        return torch.stack(outputs, dim=1).squeeze(-1)


def load_from_url(url, cache_dir='.'):
    filename = os.path.join(cache_dir, url.split('/')[-1])
    if not os.path.exists(filename):
        with tqdm(unit='B', unit_scale=True, desc=filename) as pbar:
            urllib.request.urlretrieve(url, filename, reporthook=lambda _, b, t: pbar.update(b))
    return torch.load(filename, map_location=torch.device('cpu'), weights_only=True)


TRAIN_DATA = "https://huggingface.co/datasets/puhsu/hw01-data/resolve/main/train_dataset.pt"
VAL_DATA = "https://huggingface.co/datasets/puhsu/hw01-data/resolve/main/val_dataset.pt"

if __name__ == "__main__":
    torch.manual_seed(0)

    train_dataset = torch.utils.data.TensorDataset(*map(torch.nan_to_num, load_from_url(TRAIN_DATA)))
    val_dataset = torch.utils.data.TensorDataset(*map(torch.nan_to_num, load_from_url(VAL_DATA)))

    Y_mean = train_dataset.tensors[1].mean()
    Y_std = train_dataset.tensors[1].std()
    train_dataset.tensors = (train_dataset.tensors[0], (train_dataset.tensors[1] - Y_mean) / Y_std)

    model = Trompt(n_columns=train_dataset.tensors[0].shape[1], n_prompts=128, d_model=128, n_cycles=6)
    device = torch.device('cuda:0')
    model.to(device)

    train_dl = torch.utils.data.DataLoader(train_dataset, num_workers=0, batch_size=8, shuffle=True)
    val_dl = torch.utils.data.DataLoader(val_dataset, num_workers=0, batch_size=1024)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

    EPOCHS = 5

    for e in range(1, EPOCHS + 1):
        model.train()
        for batch in tqdm(train_dl):
            x, y = batch
            opt.zero_grad()
            pred = model(x.to(device))
            loss = F.mse_loss(pred, y.unsqueeze(1).repeat(1, len(model.tcells)).to(device))
            loss.backward()
            opt.step()

        model.eval()
        mae = 0
        with torch.inference_mode():
            for batch in val_dl:
                x, y = batch
                pred = model(x.to(device))
                mae += (pred.mean(dim=-1) * Y_std + Y_mean - y.to(device)).abs().sum().item()

            mae = mae / len(val_dataset)

            print(f'>>> Epoch {e:>02}')
            print(f'Validation MAE = {mae:.5f}')
            print('>>>\n')

Writing train.py


In [3]:
%%writefile optimized_model.py
import torch
import torch.nn.functional as F
from train import Trompt as ReferenceTrompt, TromptCell as ReferenceCell, TromptDownstream


class TromptCell(ReferenceCell):
    def forward(self, x, prev_cell_out):
        x_emb = self.ln_emb(F.relu(
            x.unsqueeze(-1) * self.feature_emb_weight + self.feature_emb_bias
        ))
        prompt = self.emb_prompt
        norm = self.ln_prompt(prompt)
        if prev_cell_out.ndim == 3:
            norm = norm.expand(prev_cell_out.shape[0], -1, -1)
        prompt = self.dense_imp(torch.cat((norm, prev_cell_out), dim=-1)) + prompt
        col = self.ln_col(self.emb_column)
        with torch.autocast(device_type=x.device.type, enabled=False):
            logits = prompt.float() @ col.float().T if prompt.dtype != torch.float64 else prompt @ col.T
            mask = logits.softmax(dim=-1)
        pool = mask @ x_emb
        scale = 1 + self.dense_expand.weight[:, 0]
        bias = self.dense_expand.bias
        return pool * scale[:, None] + mask.sum(-1, keepdim=True) * bias[:, None]


class Trompt(ReferenceTrompt):
    def __init__(self, n_columns, n_prompts=128, d_model=128, n_cycles=6):
        torch.nn.Module.__init__(self)
        self.tcells = torch.nn.ModuleList([TromptCell(n_columns, n_prompts, d_model) for _ in range(n_cycles)])
        self.tdown = TromptDownstream(d_model)
        self.prompt = torch.nn.Parameter(torch.empty(n_prompts, d_model))
        self.reset_parameters()

    def forward(self, x):
        return torch.cat([self.tdown(cell(x, self.prompt)) for cell in self.tcells], dim=1)


class GemmTromptCell(TromptCell):
    def forward(self, x, prev_cell_out):
        if prev_cell_out.ndim != 2:
            return super().forward(x, prev_cell_out)
        batch, cols = x.shape
        width = self.feature_emb_weight.shape[1]
        emb = self.ln_emb(F.relu(
            x.T.unsqueeze(-1) * self.feature_emb_weight[:, None, :]
            + self.feature_emb_bias[:, None, :]
        ))
        prompt = self.dense_imp(torch.cat((self.ln_prompt(self.emb_prompt), prev_cell_out), -1)) + self.emb_prompt
        col = self.ln_col(self.emb_column)
        with torch.autocast(device_type=x.device.type, enabled=False):
            logits = prompt @ col.T if prompt.dtype == torch.float64 else prompt.float() @ col.float().T
            mask = logits.softmax(-1)
        weights = mask * (1 + self.dense_expand.weight)
        pool = weights @ emb.reshape(cols, batch * width)
        bias = mask.sum(-1, keepdim=True) * self.dense_expand.bias[:, None]
        return pool.reshape(-1, batch, width).permute(1, 0, 2) + bias


class GemmTrompt(Trompt):
    def __init__(self, n_columns, n_prompts=128, d_model=128, n_cycles=6):
        torch.nn.Module.__init__(self)
        self.tcells = torch.nn.ModuleList([GemmTromptCell(n_columns, n_prompts, d_model) for _ in range(n_cycles)])
        self.tdown = TromptDownstream(d_model)
        self.prompt = torch.nn.Parameter(torch.empty(n_prompts, d_model))
        self.reset_parameters()

Writing optimized_model.py


In [4]:
%%writefile train_optimized.py
import argparse
import contextlib
import json
import math
import os
import platform
import time
from pathlib import Path
from itertools import islice

import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset, Sampler
from tqdm import tqdm

from train import Trompt as ReferenceTrompt, load_from_url, TRAIN_DATA, VAL_DATA
from optimized_model import Trompt, GemmTrompt


class ExactDistributedSampler(Sampler):
    def __init__(self, size, rank, world_size, batch_size, seed=0, layout="balanced"):
        self.size, self.rank, self.world_size = size, rank, world_size
        self.batch_size, self.seed, self.epoch = batch_size, seed, 0
        self.layout = layout
        self.steps = math.ceil(size / (batch_size * world_size))
        if layout == "fixed" and self.steps > 1 and 0 < size % (batch_size * world_size) < world_size:
            self.steps -= 1
        while self.steps > 1 and size // self.steps < world_size:
            self.steps -= 1

    def __len__(self):
        return self.steps

    def global_count(self, step):
        if self.layout == "fixed":
            return self.size - step * self.batch_size * self.world_size if step == self.steps - 1 else self.batch_size * self.world_size
        return self.size // self.steps + int(step < self.size % self.steps)

    def tensor_batches(self):
        order = torch.randperm(self.size, generator=torch.Generator().manual_seed(self.seed + self.epoch))
        batches = torch.split(order, [self.global_count(i) for i in range(self.steps)])
        for chunk in batches:
            yield torch.tensor_split(chunk, self.world_size)[self.rank]

    def __iter__(self):
        return (idx.tolist() for idx in self.tensor_batches())


class TensorBatchLoader:
    def __init__(self, dataset, sampler):
        self.tensors, self.sampler = dataset.tensors, sampler

    def __len__(self):
        return len(self.sampler)

    def __iter__(self):
        for idx in self.sampler.tensor_batches():
            idx = idx.to(self.tensors[0].device)
            yield tuple(t.index_select(0, idx) for t in self.tensors)


class ValidationLoader:
    def __init__(self, dataset, rank, world, batch_size):
        self.tensors = tuple(t[rank::world] for t in dataset.tensors)
        self.batch_size = batch_size

    def __len__(self):
        return math.ceil(len(self.tensors[0]) / self.batch_size)

    def __iter__(self):
        for start in range(0, len(self.tensors[0]), self.batch_size):
            yield tuple(t[start:start + self.batch_size] for t in self.tensors)


def save_metrics(path, rows):
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(rows, indent=2, allow_nan=False))
    tmp.replace(path)


def sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    if dist.is_initialized():
        dist.barrier()


def reduce(tensor, op=dist.ReduceOp.SUM):
    if dist.is_initialized():
        dist.all_reduce(tensor, op=op)
    return tensor


def prepare_data(cache, validation='full', val_max_samples=1024):
    Path(cache).mkdir(parents=True, exist_ok=True)
    x, y = map(torch.nan_to_num, load_from_url(TRAIN_DATA, cache))
    x, y = x.float(), y.float().reshape(-1)
    mean, std = y.mean(), y.std()
    if not torch.isfinite(std) or std <= 0:
        raise ValueError('Invalid target standard deviation')
    valid = None
    if validation != 'none':
        vx, vy = map(torch.nan_to_num, load_from_url(VAL_DATA, cache))
        if validation == 'subset':
            vx, vy = vx[:val_max_samples], vy[:val_max_samples]
        valid = TensorDataset(vx.float(), vy.float().reshape(-1))
    return TensorDataset(x, (y - mean) / std), valid, mean, std


def train(args):
    world = int(os.environ.get('WORLD_SIZE', 1))
    rank = int(os.environ.get('RANK', 0))
    local = int(os.environ.get('LOCAL_RANK', 0))
    device = torch.device(f'cuda:{local}' if torch.cuda.is_available() and not args.cpu else 'cpu')
    if device.type == 'cuda':
        torch.cuda.set_device(device)
    if world > 1:
        dist.init_process_group('nccl' if device.type == 'cuda' else 'gloo',
                                **({'device_id': device} if device.type == 'cuda' else {}))
    def announce(message):
        if rank == 0:
            print(message, flush=True)

    announce(f'Loading data | model={args.model} | GPUs/processes={world} | validation={args.validation}')
    torch.manual_seed(args.seed)
    if args.synthetic:
        g = torch.Generator().manual_seed(args.seed)
        data = TensorDataset(torch.randn(37, 7, generator=g), torch.randn(37, generator=g))
        valid = TensorDataset(torch.randn(13, 7, generator=g), torch.randn(13, generator=g))
        mean, std = torch.tensor(0.), torch.tensor(1.)
        dims = dict(n_prompts=5, d_model=8, n_cycles=2)
    else:
        if rank != 0:
            dist.barrier()
        data, valid, mean, std = prepare_data(args.cache_dir, args.validation, args.val_max_samples)
        if world > 1 and rank == 0:
            dist.barrier()
        dims = dict(n_prompts=128, d_model=128, n_cycles=6)
    if args.validation == 'none':
        valid = None
    elif args.validation == 'subset' and args.synthetic:
        valid = TensorDataset(*(t[:args.val_max_samples] for t in valid.tensors))
    mean, std = mean.to(device), std.to(device)
    if args.data_on_gpu:
        data = TensorDataset(*(t.to(device) for t in data.tensors))
        if valid is not None:
            valid = TensorDataset(*(t.to(device) for t in valid.tensors))
    sampler = ExactDistributedSampler(len(data), rank, world, args.batch_size, args.seed, args.batch_layout)
    loader = TensorBatchLoader(data, sampler) if args.data_on_gpu else DataLoader(
        data, batch_sampler=sampler, num_workers=0, pin_memory=device.type == 'cuda')
    validloader = ValidationLoader(valid, rank, world, args.val_batch_size) if valid is not None else None
    cls = {'baseline': ReferenceTrompt, 'optimized': Trompt, 'gemm': GemmTrompt}[args.model]
    raw = cls(data.tensors[0].shape[1], **dims).to(device)
    announce('Preparing model' + (' with torch.compile; first warmup may take several minutes' if args.compile else ''))
    model = torch.compile(raw, mode=args.compile_mode, dynamic=args.compile_dynamic) if args.compile else raw
    if world > 1:
        model = DDP(model, device_ids=[local] if device.type == 'cuda' else None,
                    gradient_as_bucket_view=True, broadcast_buffers=False)
    opt = torch.optim.AdamW(raw.parameters(), lr=3e-4, weight_decay=1e-5,
                                 fused=device.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda', enabled=args.amp == 'fp16', init_scale=128.0)
    dtype = torch.float16 if args.amp == 'fp16' else torch.bfloat16
    amp = lambda: torch.autocast(device_type=device.type, dtype=dtype, enabled=args.amp != 'off')
    out = Path(args.output)
    if rank == 0:
        out.mkdir(parents=True, exist_ok=True)
        meta = dict(vars(args), torch=torch.__version__, python=platform.python_version(),
                        cuda=torch.version.cuda, world_size=world,
                        gpu=torch.cuda.get_device_name(device) if device.type == 'cuda' else None,
                        train_samples=len(data), architecture=dims)
        (out / 'config.json').write_text(json.dumps(meta, indent=2))

    def step(x, y, global_count):
        opt.zero_grad(set_to_none=True)
        with amp():
            pred = model(x)
            loss = F.mse_loss(pred.float(), y[:, None].expand_as(pred), reduction='sum') / pred.shape[1]
            objective = loss * world / global_count
        scaler.scale(objective).backward()
        scaler.unscale_(opt)
        norms = torch._foreach_norm([p.grad for p in raw.parameters() if p.grad is not None])
        finite = torch.isfinite(torch.stack(norms)).all()
        finite = finite & torch.isfinite(loss)
        if not reduce(finite.to(torch.int32), dist.ReduceOp.MIN).item():
            raise FloatingPointError('Non-finite loss or gradients; this run is invalid')
        scaler.step(opt)
        scaler.update()
        return loss.detach()

    warmup = 0.0
    if args.warmup_steps:
        initial = {k: v.detach().clone() for k, v in raw.state_dict().items()}
        wx, wy = next(iter(loader))
        wx, wy = wx.to(device), wy.to(device)
        count = sampler.global_count(0)
        announce(f'Warmup: {args.warmup_steps} steps (excluded from training measurements)')
        sync(device)
        start = time.perf_counter()
        for index in range(args.warmup_steps):
            step(wx, wy, count)
            announce(f'Warmup {index + 1}/{args.warmup_steps} complete')
        sync(device)
        warmup = time.perf_counter() - start
        raw.load_state_dict(initial)
        opt.state.clear()
        opt.zero_grad(set_to_none=True)
        scaler = torch.amp.GradScaler('cuda', enabled=args.amp == 'fp16', init_scale=128.0)
        del initial
        announce(f'Warmup finished: {warmup:.1f} s')
    history = []
    for epoch in range(1, args.epochs + 1):
        sampler.epoch = epoch
        model.train()
        total = torch.zeros((), device=device)
        seen = 0
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)
        activities = [torch.profiler.ProfilerActivity.CPU]
        if device.type == 'cuda':
            activities.append(torch.profiler.ProfilerActivity.CUDA)
        profile = torch.profiler.profile(activities=activities, record_shapes=True, profile_memory=True) if args.profile and epoch == 1 else contextlib.nullcontext()
        steps = min(args.max_steps or len(loader), len(loader))
        announce(f'Train epoch {epoch}/{args.epochs}: {steps} steps')
        sync(device)
        start = time.perf_counter()
        with profile as prof:
            for step_index, (x, y) in enumerate(tqdm(islice(loader, steps), total=steps, disable=rank != 0, desc=f'Train {epoch}', mininterval=2)):
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                global_count = sampler.global_count(step_index)
                total += step(x, y, global_count)
                seen += len(y)
        sync(device)
        elapsed = reduce(torch.tensor(time.perf_counter() - start, device=device), dist.ReduceOp.MAX).item()
        samples = int(reduce(torch.tensor(seen, device=device)).item())
        loss = reduce(total).item() / samples
        memory = torch.cuda.max_memory_allocated(device) if device.type == 'cuda' else 0
        memory = int(reduce(torch.tensor(memory, device=device), dist.ReduceOp.MAX).item())
        row = dict(epoch=epoch, train_seconds=elapsed, train_samples=samples,
                   train_samples_per_second=samples / elapsed, train_mse=loss,
                   val_mae=None, val_samples=0, validation=args.validation, validation_seconds=0.0,
                   peak_memory_bytes_per_gpu=memory, warmup_seconds=warmup,
                   full_epoch=samples == len(data), profiled=bool(args.profile and epoch == 1),
                   status='training_complete')
        history.append(row)
        if rank == 0:
            save_metrics(out / 'metrics.json', history)
            announce(f'Training finished: {samples / elapsed:.2f} samples/s, {elapsed:.2f} s')
        if prof is not None:
            out.mkdir(parents=True, exist_ok=True)
            announce('Saving profiler trace')
            prof.export_chrome_trace(str(out / f'trace-rank{rank}.json'))
        if validloader is not None:
            announce(f'Validation: {len(valid)} samples across {world} processes')
            raw.eval()
            error = torch.zeros((), device=device)
            sync(device)
            started = time.perf_counter()
            with torch.inference_mode():
                for x, y in tqdm(validloader, disable=rank != 0, desc=f'Validation {epoch}', mininterval=2):
                    x, y = x.to(device), y.to(device)
                    with amp():
                        pred = raw(x)
                    error += (pred.float().mean(-1) * std + mean - y).abs().sum()
            mae = reduce(error).item() / len(valid)
            sync(device)
            row.update(val_mae=mae, val_samples=len(valid), validation_seconds=time.perf_counter() - started)
            if not math.isfinite(mae):
                raise FloatingPointError('Non-finite validation MAE; this run is invalid')
        row['status'] = 'complete'
        if rank == 0:
            if args.checkpoint:
                announce('Saving checkpoint')
                torch.save(dict(model=raw.state_dict(), optimizer=opt.state_dict(),
                                scaler=scaler.state_dict(), epoch=epoch, y_mean=mean.cpu(), y_std=std.cpu()),
                           out / 'checkpoint.pt')
            save_metrics(out / 'metrics.json', history)
            announce(json.dumps(row))
    announce(f'DONE: {out / "metrics.json"}')
    if world > 1:
        dist.destroy_process_group()


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--model', choices=['baseline', 'optimized', 'gemm'], default='optimized')
    p.add_argument('--batch-size', type=int, default=128, help='Maximum batch size per GPU')
    p.add_argument('--val-batch-size', type=int, default=128)
    p.add_argument('--validation', choices=['full', 'subset', 'none'], default='full')
    p.add_argument('--val-max-samples', type=int, default=1024)
    p.add_argument('--checkpoint', action=argparse.BooleanOptionalAction, default=True)
    p.add_argument('--epochs', type=int, default=5)
    p.add_argument('--amp', choices=['off', 'fp16', 'bf16'], default='off')
    p.add_argument('--compile', action='store_true')
    p.add_argument('--compile-dynamic', action=argparse.BooleanOptionalAction, default=True)
    p.add_argument('--batch-layout', choices=['balanced', 'fixed'], default='balanced')
    p.add_argument('--compile-mode', default='default', choices=['default', 'reduce-overhead', 'max-autotune'])
    p.add_argument('--data-on-gpu', action='store_true')
    p.add_argument('--warmup-steps', type=int, default=3)
    p.add_argument('--max-steps', type=int, default=0, help='0 means a complete epoch')
    p.add_argument('--profile', action='store_true')
    p.add_argument('--cache-dir', default='data')
    p.add_argument('--output', default='results/optimized')
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--cpu', action='store_true')
    p.add_argument('--synthetic', action='store_true')
    args = p.parse_args()
    return args


if __name__ == '__main__':
    train(parse_args())

Writing train_optimized.py


In [5]:
%%writefile notebook_runner.py
import os
from pathlib import Path
import queue
import signal
import subprocess
import threading
import time


def run_logged(command, log_path, timeout_seconds=1800, heartbeat_seconds=20, cwd=None):
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    start = time.monotonic()
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=env, start_new_session=True, cwd=cwd)
    queuebuf = queue.Queue()

    def read_output():
        try:
            for line in proc.stdout:
                queuebuf.put(line)
        finally:
            queuebuf.put(None)

    reader = threading.Thread(target=read_output, daemon=True)
    reader.start()

    def stop():
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGTERM)
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL)
                proc.wait()

    try:
        with path.open('w', buffering=1) as log:
            while True:
                elapsed = time.monotonic() - start
                if elapsed >= timeout_seconds:
                    raise TimeoutError(f'Run exceeded {timeout_seconds}s; see {path}')
                try:
                    line = queuebuf.get(timeout=min(heartbeat_seconds, timeout_seconds - elapsed))
                except queue.Empty:
                    print(f'Process running: {time.monotonic() - start:.0f}s elapsed | log: {path}', flush=True)
                    continue
                if line is None:
                    break
                log.write(line)
                print(line, end='', flush=True)
        code = proc.wait(timeout=max(0.1, timeout_seconds - (time.monotonic() - start)))
        if code:
            raise subprocess.CalledProcessError(code, command)
        print(f'Completed in {time.monotonic() - start:.1f}s | log: {path}', flush=True)
    finally:
        stop()
        reader.join(timeout=2)
        proc.stdout.close()

Writing notebook_runner.py


In [6]:
%%writefile tune_trompt.py
import argparse
import json
from pathlib import Path
import sys

from notebook_runner import run_logged


def score(rows):
    valid = [r for r in rows if r['status'] == 'complete' and not r['profiled']]
    return sum(r['train_samples'] for r in valid) / sum(r['train_seconds'] for r in valid)


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--cache-dir', default='/kaggle/working/data')
    p.add_argument('--output', default='results_v3')
    p.add_argument('--steps', type=int, default=20)
    p.add_argument('--target', type=float, default=7000)
    args = p.parse_args()
    root = Path(args.output)
    root.mkdir(exist_ok=True)
    configs = [
        dict(model='gemm', batch=512, compile=False),
        dict(model='gemm', batch=1024, compile=False),
        dict(model='gemm', batch=2048, compile=False),
        dict(model='gemm', batch=1024, compile=True),
        dict(model='gemm', batch=2048, compile=True),
    ]
    results = []
    for cfg in configs:
        name = f"{cfg['model']}_b{cfg['batch']}_{'compiled' if cfg['compile'] else 'eager'}"
        dest = root / name
        log = root / 'logs' / f'{name}.txt'
        opts = ['--model', cfg['model'], '--batch-size', str(cfg['batch']),
                   '--batch-layout', 'fixed', '--no-compile-dynamic', '--amp', 'fp16', '--data-on-gpu',
                   '--epochs', '1', '--max-steps', str(args.steps), '--validation', 'none',
                   '--no-checkpoint', '--warmup-steps', '3', '--cache-dir', args.cache_dir,
                   '--output', str(dest)]
        if cfg['compile']:
            opts += ['--compile']
        cmd = [sys.executable, '-u', '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
                   'train_optimized.py', *opts]
        print(f'Candidate: {name}', flush=True)
        run_logged(cmd, log, timeout_seconds=600)
        rows = json.loads((dest / 'metrics.json').read_text())
        result = dict(cfg, name=name, options=opts, status='complete',
                      samples_per_second=score(rows),
                      peak_memory_bytes=max(r['peak_memory_bytes_per_gpu'] for r in rows))
        results.append(result)
        (root / 'tuning.json').write_text(json.dumps(results, indent=2))
        print(json.dumps({k: v for k, v in result.items() if k != 'options'}), flush=True)
    winner = max(results, key=lambda r: r['samples_per_second'])
    (root / 'selected.json').write_text(json.dumps(winner, indent=2))
    print(f"Selected {winner['name']}: {winner['samples_per_second']:.1f} samples/s (short measurement)", flush=True)
    if winner['samples_per_second'] < args.target:
        print('Target not reached in short runs. The optimized profile is required for further diagnosis.', flush=True)


if __name__ == '__main__':
    main()

Writing tune_trompt.py


In [7]:
%%writefile tests/test_notebook_runner.py
import subprocess
import sys

import pytest
from notebook_runner import run_logged


def test_live_output_and_log(tmp_path, capsys):
    path = tmp_path / 'run.txt'
    run_logged([sys.executable, '-u', '-c', 'print("batch 1"); print("finished")'], path, timeout_seconds=10)
    assert 'batch 1' in capsys.readouterr().out
    assert path.read_text() == 'batch 1\nfinished\n'


def test_child_failure_is_not_hidden(tmp_path):
    with pytest.raises(subprocess.CalledProcessError):
        run_logged([sys.executable, '-c', 'raise RuntimeError("failure")'], tmp_path / 'error.txt', timeout_seconds=10)
    assert 'failure' in (tmp_path / 'error.txt').read_text()


def test_timeout_stops_process(tmp_path):
    with pytest.raises(TimeoutError):
        run_logged([sys.executable, '-c', 'import time; time.sleep(60)'], tmp_path / 'timeout.txt',
                   timeout_seconds=0.3, heartbeat_seconds=0.1)

Writing tests/test_notebook_runner.py


In [8]:
%%writefile tests/test_optimized.py
import pytest
import torch
from train import Trompt as Reference
from optimized_model import Trompt, GemmTrompt


@pytest.mark.parametrize('device', ['cpu'] + (['cuda'] if torch.cuda.is_available() else []))
@pytest.mark.parametrize('model_class', [Trompt, GemmTrompt])
@pytest.mark.parametrize('dtype', [torch.float64, torch.float32])
@pytest.mark.parametrize('batch', [1, 4])
def test_outputs_and_all_gradients(dtype, batch, model_class, device):
    torch.manual_seed(12)
    slow = Reference(7, 5, 8, 3).to(device=device, dtype=dtype)
    fast = model_class(7, 5, 8, 3).to(device=device, dtype=dtype)
    fast.load_state_dict(slow.state_dict())
    x1 = torch.randn(batch, 7, device=device, dtype=dtype, requires_grad=True)
    x2 = x1.detach().clone().requires_grad_(True)
    a, b = slow(x1), fast(x2)
    tol = dict(atol=2e-6, rtol=2e-5) if dtype == torch.float32 else dict(atol=1e-10, rtol=1e-9)
    torch.testing.assert_close(a, b, **tol)
    target = torch.randn_like(a)
    (a - target).square().mean().backward()
    (b - target).square().mean().backward()
    grad_tol = dict(atol=1e-5, rtol=1e-4) if dtype == torch.float32 else tol
    torch.testing.assert_close(x1.grad, x2.grad, **grad_tol)
    for (name, p), (name2, q) in zip(slow.named_parameters(), fast.named_parameters()):
        assert name == name2 and p.grad is not None and q.grad is not None
        torch.testing.assert_close(p.grad, q.grad, msg=lambda detail: f"{name}\n{detail}", **grad_tol)


@pytest.mark.parametrize('model_class', [Trompt, GemmTrompt])
def test_optimizer_updates_match(model_class):
    torch.manual_seed(24)
    slow, fast = Reference(5, 4, 8, 2).double(), model_class(5, 4, 8, 2).double()
    fast.load_state_dict(slow.state_dict())
    opts = [torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=1e-5) for m in (slow, fast)]
    for _ in range(3):
        x, y = torch.randn(4, 5, dtype=torch.float64), torch.randn(4, 2, dtype=torch.float64)
        for m, opt in zip((slow, fast), opts):
            opt.zero_grad()
            loss = (m(x) - y).square().mean()
            assert torch.isfinite(loss)
            loss.backward()
            opt.step()
    for p, q in zip(slow.parameters(), fast.parameters()):
        torch.testing.assert_close(p, q, atol=2e-5, rtol=2e-4)

Writing tests/test_optimized.py


In [9]:
%%writefile tests/test_sampler.py
import pytest
from train_optimized import ExactDistributedSampler


@pytest.mark.parametrize('layout', ['balanced', 'fixed'])
@pytest.mark.parametrize('size,world,batch', [(37, 2, 8), (5, 2, 1), (160001, 2, 128), (7, 3, 2)])
def test_ddp_counts_and_coverage(size, world, batch, layout):
    ranks = [list(ExactDistributedSampler(size, rank, world, batch, layout=layout)) for rank in range(world)]
    assert len({len(batches) for batches in ranks}) == 1
    seen = [i for batches in ranks for idx in batches for i in idx]
    assert sorted(seen) == list(range(size))
    assert all(len(idx) > 0 for batches in ranks for idx in batches)


@pytest.mark.parametrize('size,world,batch', [(37, 2, 8), (5, 2, 1), (160001, 2, 1024)])
def test_sampler_counts_match_real_batches(size, world, batch):
    sampler = ExactDistributedSampler(size, 0, world, batch, layout='fixed')
    ranks = [list(ExactDistributedSampler(size, rank, world, batch, layout='fixed')) for rank in range(world)]
    assert [sum(len(ranks[r][i]) for r in range(world)) for i in range(len(sampler))] == [sampler.global_count(i) for i in range(len(sampler))]

Writing tests/test_sampler.py


In [10]:
%%writefile tests/test_training_cli.py
import json
from pathlib import Path
import subprocess
import sys

import pytest
import torch
from train_optimized import ExactDistributedSampler, TensorBatchLoader, ValidationLoader, prepare_data

ROOT = Path(__file__).resolve().parents[1]


def test_no_validation_does_not_load_validation_data(monkeypatch, tmp_path):
    import train_optimized as training
    calls = []
    def load(url, cache):
        calls.append(url)
        return torch.zeros(9, 3), torch.arange(9).float()
    monkeypatch.setattr(training, 'load_from_url', load)
    train, val, mean, std = prepare_data(tmp_path, 'none')
    assert calls == [training.TRAIN_DATA] and val is None and len(train) == 9


def test_tensor_loader_matches_sampler_and_batch_counts():
    data = torch.utils.data.TensorDataset(torch.arange(37)[:, None], torch.arange(37))
    for rank in range(2):
        sampler = ExactDistributedSampler(37, rank, 2, 8)
        expected = list(sampler)
        actual = list(TensorBatchLoader(data, sampler))
        assert [y.tolist() for _, y in actual] == expected
    s = ExactDistributedSampler(37, 0, 2, 8)
    assert sum(s.global_count(i) for i in range(len(s))) == 37
    vals = [list(ValidationLoader(data, rank, 2, 4)) for rank in range(2)]
    assert sorted(i for rank in vals for _, y in rank for i in y.tolist()) == list(range(37))


@pytest.mark.parametrize('validation,extra,expected_samples', [
    ('none', ['--model', 'baseline', '--profile', '--max-steps', '3', '--no-checkpoint'], 0),
    ('full', ['--data-on-gpu'], 13),
    ('subset', ['--val-max-samples', '5'], 5),
])
def test_training_modes(tmp_path, validation, extra, expected_samples):
    cmd = [sys.executable, '-u', 'train_optimized.py', '--cpu', '--synthetic',
               '--epochs', '1', '--batch-size', '8', '--validation', validation,
               '--output', str(tmp_path), *extra]
    result = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, timeout=45)
    assert result.returncode == 0, result.stdout + result.stderr
    row = json.loads((tmp_path / 'metrics.json').read_text())[0]
    assert row['status'] == 'complete' and row['val_samples'] == expected_samples
    assert (row['val_mae'] is None) == (validation == 'none')
    assert row['full_epoch'] == ('--max-steps' not in extra)
    if validation == 'none':
        assert row['train_samples'] == 23
        assert (tmp_path / 'trace-rank0.json').exists()
        assert not (tmp_path / 'checkpoint.pt').exists()
        assert 'Validation:' not in result.stdout
    else:
        assert row['train_samples'] == 37
        assert (tmp_path / 'checkpoint.pt').exists()
    assert 'DONE:' in result.stdout

Writing tests/test_training_cli.py


In [11]:
from notebook_runner import run_logged
import torch
run_logged([sys.executable, "-m", "pip", "install", "-q", "pytest", "tqdm", "pandas"], "logs_v3/install.txt", timeout_seconds=180)
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(2)])
run_logged([sys.executable, "-m", "pytest", "tests", "-q"], "logs_v3/tests.txt", timeout_seconds=240)

Completed in 4.0s | log: logs_v3/install.txt
2.10.0+cu128 ['Tesla T4', 'Tesla T4']
.....................................                                    [100%]
37 passed in 18.20s
Completed in 20.7s | log: logs_v3/tests.txt


## Изменения

Мелкие операции объединены в более крупные. Убраны лишние копии тензоров. Общая матрица используется сразу для всего батча. Архитектура модели сохранена.

In [12]:
cache = str(base / "data")
run_logged([sys.executable, "-u", "tune_trompt.py", "--cache-dir", cache],
           "logs_v3/tuning.txt", timeout_seconds=1800)
selected = json.loads(Path("results_v3/selected.json").read_text())
print("Selected:", selected["name"], selected["samples_per_second"])

Candidate: gemm_b512_eager

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W916 07:56:39.663700676 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 07:56:42.781451003 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 07:56:42.863537131 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Loading data | model=gemm | GPUs/processes=2 | validation=none

/kaggle/working/data/train_dataset.pt: 0.00B [00:00, ?B/s]
/kaggle/working/data/train_dataset.pt: 8.19kB [00:00, 38.3kB/s]
/kaggle/working/data/train_dataset.pt: 16.4kB [00:00, 25.3kB/s]
/kaggle/working/data/train_dataset.pt: 9.62MB [00:00, 18.0MB/s]
/kaggle/working/data/train_datase

In [13]:
def command(output, epochs, validation, max_steps=0, profile=False):
    command = [sys.executable, "-u", "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=2",
               "train_optimized.py", "--model", selected["model"], "--batch-size", str(selected["batch"]),
               "--val-batch-size", "256", "--batch-layout", "fixed", "--no-compile-dynamic",
               "--amp", "fp16", "--data-on-gpu", "--epochs", str(epochs), "--validation", validation,
               "--max-steps", str(max_steps), "--cache-dir", cache, "--output", output]
    if selected["compile"]:
        command += ["--compile"]
    if profile:
        command += ["--profile", "--no-checkpoint"]
    return command

run_logged(command("results_v3/optimized_profile", 1, "none", 3, True),
           "logs_v3/optimized_profile.txt", timeout_seconds=600)


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W916 08:00:20.412762932 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 08:00:22.557406490 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 08:00:22.575503085 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Loading data | model=gemm | GPUs/processes=2 | validation=none
Preparing model with torch.compile; first warmup may take several minutes
Warmup: 3 steps (excluded from training measurements)
Warmup 1/3 complete
Warmup 2/3 complete
Warmup 3/3 complete
Warmup finished: 11.6 s
Train epoch 1/1: 3 steps
/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: Use

## Измерение

Профиль нужен для поиска медленных операций. Итоговая скорость считается по полным эпохам без профилировщика. Прогрев и валидация учитываются отдельно.

In [14]:
run_logged(command("results_v3/final", 5, "full"),
           "logs_v3/final.txt", timeout_seconds=1800)


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W916 08:02:06.820345382 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 08:02:08.051697026 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 08:02:08.090188815 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Loading data | model=gemm | GPUs/processes=2 | validation=full

/kaggle/working/data/val_dataset.pt: 0.00B [00:00, ?B/s]
/kaggle/working/data/val_dataset.pt: 8.19kB [00:00, 39.9kB/s]
/kaggle/working/data/val_dataset.pt: 16.4kB [00:00, 24.1kB/s]
/kaggle/working/data/val_dataset.pt: 7.20MB [00:00, 14.2MB/s]
/kaggle/working/data/val_dataset.pt: 31.9MB [00:00, 65.7MB/s]
/kaggl

In [15]:
final = json.loads(Path("results_v3/final/metrics.json").read_text())
assert len(final) == 5
assert all(r["status"] == "complete" and r["full_epoch"] and r["validation"] == "full" for r in final)
speed = sum(r["train_samples"] for r in final) / sum(r["train_seconds"] for r in final)
steady = final[1:]
rate = sum(r["train_samples"] for r in steady) / sum(r["train_seconds"] for r in steady)
print("All 5 full epochs:", round(speed, 1), "samples/s")
print("Full epochs 2–5:", round(rate, 1), "samples/s")
print("Final MAE:", final[-1]["val_mae"])
print("Target reached over all epochs:", speed >= 7000)
import pandas as pd
display(pd.DataFrame(final))

All 5 full epochs: 7719.3 samples/s
Full epochs 2–5: 13361.4 samples/s
Final MAE: 0.12732484563880783
Target reached over all epochs: True


,epoch,train_seconds,train_samples,train_samples_per_second,train_mse,val_mae,val_samples,validation,validation_seconds,peak_memory_bytes_per_gpu,warmup_seconds,full_epoch,profiled,status
0,1,55.743347,160019,2870.638527,0.355997,0.137096,59975,full,5.831432,11046281728,11.352582,True,False,complete
1,2,11.760495,160019,13606.484886,0.227768,0.131134,59975,full,5.867516,11046282752,11.352582,True,False,complete
2,3,11.881402,160019,13468.023369,0.213339,0.128954,59975,full,5.959537,11046282752,11.352582,True,False,complete
3,4,12.029324,160019,13302.410478,0.206772,0.127521,59975,full,6.039949,11046282752,11.352582,True,False,complete
4,5,12.233618,160019,13080.268065,0.203066,0.127325,59975,full,6.118715,11046282752,11.352582,True,False,complete


## Выводы

Объединение операций и компиляция заметно ускорили обучение. Данные обрабатываются на обеих видеокартах. Ошибка уменьшается, а проверки позволяют контролировать правильность вычислений. 

In [16]:
dest = base / "training_results_v3.zip"
with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in ("results_v3", "logs_v3"):
        for path in Path(folder).rglob("*"):
            if path.is_file():
                archive.write(path, path)
print(dest)

/kaggle/working/training_results_v3.zip
